# Brain playground

Interactive poking around at my scans. Everything heavy lives in the pipeline (`python run_pipeline.py`);
this notebook is for looking.

* **Part 1** works straight from `nifti/`, with nothing else needed.
* **Part 2** uses the pipeline outputs in `derivatives/`, `results/` and `figures/`.

In [ ]:
%matplotlib inline
import json, sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd()))          # run from the repo root
import numpy as np, nibabel as nib, pandas as pd
import matplotlib.pyplot as plt
from nilearn import plotting, image
from mybrain.config import NIFTI, RESULTS, FIGURES, RUNS, TR, t1, bold

## Part 1: the raw scans

Every NIfTI in `nifti/`. 3D scans are shown directly; for 4D fMRI runs we show the mean over time.

In [ ]:
four_d = []
for f in sorted(NIFTI.glob('*.nii')):
    img = nib.load(f)
    if img.ndim == 4:
        four_d.append(f)
        img = image.mean_img(img, copy_header=True)
    print(f.name, nib.load(f).shape)
    plotting.plot_img(img, title=f.stem[:40], display_mode='ortho', cmap='gray')
    plt.show()

Scroll through my T1 anatomy in 3D (this is what the old `3d_brain.html` showed):

In [ ]:
plotting.view_img(str(t1()), bg_img=False, cmap='gray', symmetric_cmap=False, threshold=None, black_bg=True)

And the mean of one fMRI run (the old `4d_brain.html`). Note the blur and the dark, squashed frontal lobe: that's what fMRI looks like before alignment.

In [ ]:
plotting.view_img(image.mean_img(str(bold(2)), copy_header=True), bg_img=False, cmap='gray', symmetric_cmap=False, threshold=None, black_bg=True)

## Part 2: pipeline results

Run `python run_pipeline.py` first.

### Data quality

In [ ]:
pd.DataFrame(json.load(open(RESULTS / 'qc.json'))).T

### When did I fall asleep?

The grey area shows where the drowsiness index is above my average for the session.

In [ ]:
display(pd.DataFrame(json.load(open(RESULTS / 'drowsiness.json'))).T)
plt.figure(figsize=(12, 12)); plt.imshow(plt.imread(FIGURES / 'drowsiness.png')); plt.axis('off');

### Networks

The connectome and the run-to-run similarity, which works like a fingerprint:

In [ ]:
net = json.load(open(RESULTS / 'networks.json'))
print('within-network r:', net['within_network_r']); print('between-network r:', net['between_network_r'])
pd.DataFrame(net['run_to_run_connectome_similarity'], index=[f'run {r}' for r in RUNS], columns=[f'run {r}' for r in RUNS])

### Make your own seed map

Pick any MNI coordinate and see which parts of my brain move with it. Try `(0, -52, 26)` (default mode), `(0, -88, 6)` (visual), or `(-22, -4, -18)` (left amygdala).

In [ ]:
from mybrain import data, space

def seed_map(xyz, radius=6, threshold=0.25):
    sel = space.sphere(xyz, radius)
    if not sel.any():
        raise ValueError('no grey-matter voxels near that coordinate')
    zs = []
    for r in RUNS:
        v = data.clean(r)[data.keep(r)]
        s = v[:, sel].mean(1); s = (s - s.mean()) / s.std()
        zs.append(np.arctanh(np.clip((v * s[:, None]).mean(0) / (v.std(0) + 1e-9), -.999, .999)))
    return plotting.view_img(space.unmask(np.mean(zs, 0)), bg_img=space.t1_in_mni(), threshold=threshold, vmax=1, cut_coords=xyz, black_bg=True)

seed_map((0, -52, 26))